# Pipeline ANVISA - Etapa 2 (Processamento da Base)

Continuidade do notebook `1_download_anvisa.ipynb`.

Fluxo coberto neste notebook:
1. Etapa 1.5 - processamento e engenharia da base consolidada
2. Etapa 2B - processamento avan?ado e exporta??es finais


In [ ]:
import os
import sys
import time
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pipelines').exists():
    raise RuntimeError('Abra este notebook na raiz do projeto (onde existe a pasta pipelines).')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 180)

print(f'Projeto: {PROJECT_ROOT}')


In [ ]:
from pipelines.anvisa_base.config_anvisa import (
    ARQUIVO_CONSOLIDADO_TEMP,
    ARQUIVO_FINAL_VIGENCIAS,
)
from pipelines.anvisa_base.workflows.stage15_processamento_engenharia import (
    main as run_stage15_processing,
)
from pipelines.anvisa_base.src.processar_dados import (
    main as run_stage2b_processing,
)
from pipelines.anvisa_base.src.config import (
    ARQUIVO_ENTRADA as ARQUIVO_ENTRADA_2B,
    ARQUIVO_SAIDA as ARQUIVO_SAIDA_2B,
)


In [ ]:
# Parametros de execucao
RUN_STAGE15 = True
RUN_STAGE2B = True

print(f'RUN_STAGE15={RUN_STAGE15}')
print(f'RUN_STAGE2B={RUN_STAGE2B}')
print('')
print(f'Entrada esperada da etapa 1.5: {Path(ARQUIVO_CONSOLIDADO_TEMP).resolve()}')
print(f'Saida da etapa 1.5: {Path(ARQUIVO_FINAL_VIGENCIAS).resolve()}')
print(f'Entrada da 2B: {Path(ARQUIVO_ENTRADA_2B).resolve()}')
print(f'Saida da 2B: {Path(ARQUIVO_SAIDA_2B).resolve()}')


In [ ]:
# Pre-check de continuidade da etapa 1
if not Path(ARQUIVO_CONSOLIDADO_TEMP).exists():
    raise FileNotFoundError(
        f'Arquivo consolidado nao encontrado: {ARQUIVO_CONSOLIDADO_TEMP}. '
        'Execute antes o notebook 1_download_anvisa.ipynb.'
    )

print('[OK] Consolidado bruto encontrado. Etapa 2 pode iniciar.')


In [ ]:
# 1) Etapa 1.5 - Processamento e engenharia
if RUN_STAGE15:
    t0 = time.time()
    run_stage15_processing()
    print(f'\n[OK] Etapa 1.5 concluida em {(time.time() - t0):.1f}s')
else:
    print('Etapa 1.5 ignorada (RUN_STAGE15=False).')


In [ ]:
# Checkpoint apos 1.5
saida_15 = Path(ARQUIVO_FINAL_VIGENCIAS)
saida_base = Path(ARQUIVO_ENTRADA_2B)

for p in [saida_15, saida_base]:
    status = 'OK' if p.exists() else 'AUSENTE'
    tamanho_mb = (p.stat().st_size / 1024 / 1024) if p.exists() else 0
    print(f'[{status}] {p} ({tamanho_mb:.2f} MB)')

if saida_base.exists():
    preview_15 = pd.read_csv(saida_base, sep=';', nrows=5, low_memory=False)
    display(preview_15)


In [ ]:
# 2) Etapa 2B - Processamento avancado
if RUN_STAGE2B:
    if not Path(ARQUIVO_ENTRADA_2B).exists():
        raise FileNotFoundError(
            f'Entrada da 2B nao encontrada: {ARQUIVO_ENTRADA_2B}. '
            'Execute a etapa 1.5 primeiro.'
        )

    t0 = time.time()
    run_stage2b_processing()
    print(f'\n[OK] Etapa 2B concluida em {(time.time() - t0):.1f}s')
else:
    print('Etapa 2B ignorada (RUN_STAGE2B=False).')


In [ ]:
# Checkpoint final das saidas da etapa 2
arquivos_esperados = [
    'output/anvisa/baseANVISA.csv',
    'output/anvisa/baseANVISA_dtypes.json',
    'output/anvisa/dfprodutos.csv',
    'output/anvisa/dfpro_correcao_manual.xlsx',
    'output/anvisa/principios_ativos_unicos.txt',
    'output/anvisa/produtos_unicos.txt',
]

status_rows = []
for rel_path in arquivos_esperados:
    p = Path(rel_path)
    status_rows.append({
        'arquivo': rel_path,
        'existe': p.exists(),
        'tamanho_mb': round((p.stat().st_size / 1024 / 1024), 2) if p.exists() else None,
    })

status_df = pd.DataFrame(status_rows)
display(status_df)

base_final = Path('output/anvisa/baseANVISA.csv')
if base_final.exists():
    preview_final = pd.read_csv(base_final, sep=';', nrows=5, low_memory=False)
    print('\nPreview da base final:')
    display(preview_final)


## Observacao operacional

- Uso padrao: mantenha `RUN_STAGE15=True` e `RUN_STAGE2B=True` para reproduzir o comportamento do `2_processar_base_anvisa.py`.
- Reprocessamento rapido: use `RUN_STAGE15=False` e `RUN_STAGE2B=True` quando quiser rerodar apenas o refinamento avancado (equivalente ao `2b_processar_dados_anvisa.py`).
